In [1]:
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info
from config import draft_mllm_name, prompt
import torch

draft_mllm = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    draft_mllm_name,
    dtype=torch.bfloat16,
    # attn_implementation="flash_attention_2",
    device_map="auto",
)

min_pixels = 256*28*28
max_pixels = 1280*28*28
processor = AutoProcessor.from_pretrained(draft_mllm_name, min_pixels=min_pixels, max_pixels=max_pixels)

tokenizer = processor.tokenizer

prompt = 

messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "image",
                "image": "https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen-VL/assets/demo.jpeg",
            },
            {"type": "text", "text": prompt},
        ],
    }
]

# Preparation for inference
text = processor.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
image_inputs, video_inputs = process_vision_info(messages)
inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt",
).to("cuda")

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

In [4]:
generated_ids = draft_mllm.generate(**inputs, max_new_tokens=128)
generated_ids, generated_ids.shape

(tensor([[151644,   8948,    198,  ...,  82112,     13, 151645]],
        device='cuda:0'),
 torch.Size([1, 1380]))

In [5]:
inputs.input_ids, inputs.input_ids.shape

(tensor([[151644,   8948,    198,  ..., 151644,  77091,    198]],
        device='cuda:0'),
 torch.Size([1, 1272]))

In [6]:
for gid, iid in zip(generated_ids, inputs.input_ids):
    print(gid, iid)
    break

tensor([151644,   8948,    198,  ...,  82112,     13, 151645], device='cuda:0') tensor([151644,   8948,    198,  ..., 151644,  77091,    198], device='cuda:0')


In [10]:
generated_ids_trimmed = [
    out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]
print(generated_ids_trimmed)
print(generated_ids_trimmed[0].shape)

[tensor([   785,   2168,  61891,    264,  94763,  11321,   6109,    448,    264,
          1697,    323,    264,   5562,  11699,    389,    279,   9278,     13,
           576,   1697,    374,  12233,    264,    625,   3779,  15478,    323,
         35776,     11,    323,    807,   4994,    311,    387,  36063,   1393,
         44730,    448,    279,   5562,     13,    576,   5562,     11,    892,
          5868,   1075,    264,  79276,  10392,    461,   2054,     11,    374,
         12233,    264,  32408,    323,    374,   1083,  36063,   1182,    518,
           279,   1697,     13,    576,   4004,   4933,    279,  17951,    448,
         21700,  16876,  45474,   8630,    279,  30184,     11,    323,    279,
         12884,    374,   2797,    448,    264,   8413,   3100,  22561,   2987,
          4124,   6556,    476,   3309,  13354,     13,    576,   8084,  16566,
           315,    279,   2168,    374,  25650,    323,  82112,     13, 151645],
       device='cuda:0')]
torch.Size([1

In [13]:
output_text = processor.batch_decode(
    generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
)
print(output_text)

[tensor([151644,   8948,    198,  ...,  82112,     13, 151645], device='cuda:0')]

In [2]:
out = draft_mllm(inputs.input_ids)

In [3]:
logits = out.logits

In [4]:
logits

tensor([[[11.2500, 19.2500, 18.5000,  ...,  8.3125,  8.3125,  8.3125],
         [10.0000, 15.0000,  9.1875,  ...,  6.8750,  6.8750,  6.8750],
         [15.3125, 18.2500, 20.6250,  ...,  6.8438,  6.8438,  6.8438],
         ...,
         [ 6.0625,  4.5000,  0.1387,  ...,  0.8984,  0.8984,  0.8945],
         [17.6250, 17.1250, 13.3750,  ...,  6.6562,  6.6562,  6.6562],
         [16.3750, 17.6250, 15.5000,  ...,  6.5000,  6.5000,  6.5000]]],
       device='cuda:0', dtype=torch.bfloat16, grad_fn=<UnsafeViewBackward0>)

In [5]:
import torch

In [8]:
probs = logits[0, -1, :].softmax(0)

In [31]:
sample = torch.multinomial(probs, num_samples=1)


In [32]:
sample

tensor([785], device='cuda:0')

In [33]:
processor.tokenizer.decode(sample)

'The'

In [34]:
logits

tensor([[[11.2500, 19.2500, 18.5000,  ...,  8.3125,  8.3125,  8.3125],
         [10.0000, 15.0000,  9.1875,  ...,  6.8750,  6.8750,  6.8750],
         [15.3125, 18.2500, 20.6250,  ...,  6.8438,  6.8438,  6.8438],
         ...,
         [ 6.0625,  4.5000,  0.1387,  ...,  0.8984,  0.8984,  0.8945],
         [17.6250, 17.1250, 13.3750,  ...,  6.6562,  6.6562,  6.6562],
         [16.3750, 17.6250, 15.5000,  ...,  6.5000,  6.5000,  6.5000]]],
       device='cuda:0', dtype=torch.bfloat16, grad_fn=<UnsafeViewBackward0>)

In [39]:
log_prob = torch.log_softmax(logits[:, -1, :], dim=-1)

In [40]:
topk_log_prob, topk_ids = log_prob.topk(3, dim=-1)

In [41]:
topk_log_prob, topk_ids

(tensor([[-0.2617, -2.1406, -2.6406]], device='cuda:0', dtype=torch.bfloat16,
        grad_fn=<TopkBackward0>),
 tensor([[  40,  785, 1986]], device='cuda:0'))

In [2]:
import torch
def beam_search(model,
                input_ids,
                beam_size,
                max_length,
                eos_token_id=None,
                device='cuda'):
    input_ids = input_ids.to(device)

    beams = [(input_ids, 0.0)]  # (sequence, score)
    completed_beams = []

    for step in range(max_length):
        new_beams = []
        for seq, score in beams:
            if eos_token_id is not None and seq[0, -1].item() == eos_token_id:
                completed_beams.append((seq, score))
                continue

            with torch.no_grad():
                outputs = model(seq)
                next_token_logits = outputs.logits[:, -1, :]
                log_probs = torch.log_softmax(next_token_logits, dim=-1)
                topk_log_probs, topk_indices = torch.topk(log_probs, beam_size, dim=-1)

                for i in range(beam_size):
                    new_token = topk_indices[0, i].unsqueeze(0).unsqueeze(0)
                    new_seq = torch.cat([seq, new_token], dim=-1)
                    new_score = score + topk_log_probs[0, i].item()

                    new_beams.append((new_seq, new_score))
        new_beams.sort(key=lambda x: x[1], reverse=True)
        beams = new_beams[:beam_size]

        if len(completed_beams) >= beam_size:
            break

    if len(completed_beams) == 0:
        completed_beams = beams
    
    completed_beams.sort(key=lambda x: x[1], reverse=True)
    best_seq, best_score = completed_beams[0]
    return best_seq, best_score

In [3]:
from config import eos_tokens_id
best_seq, best_score = beam_search(draft_mllm, inputs.input_ids, 3, 20, eos_tokens_id)

In [4]:
best_seq

tensor([[151644,   8948,    198,  ...,    311,     13, 151645]],
       device='cuda:0')

In [5]:
best_seq[0][len(inputs.input_ids[0]):]

tensor([    40,   2776,  14589,     11,    714,    358,    646,    944,   7512,
           279,   2168,    498,   2299,  22023,    311,     13, 151645],
       device='cuda:0')

In [6]:
processor.tokenizer.decode(best_seq[0][len(inputs.input_ids[0]):], skip_special_tokens= True)

"I'm sorry, but I can't describe the image you're referring to."